# **Thesis - Predictive Processing Component**
## ADNI MERGE - ALZHEIMER Dataset

# 0. Imports

In [1]:
import sys

In [2]:
print(sys.executable)

c:\Users\maria\OneDrive - NOVAIMS\Documents\GitHub\ADNI-MERGE\p_adnimerge\Scripts\python.exe


In [3]:
# General setup & utilities
import warnings
from math import ceil
import sys


# Data Manipulation & loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [4]:
# Data
original_data = pd.read_csv('data/ADNIMERGE.csv', low_memory=False, na_values=[""])

In [5]:
original_data

,RID,COLPROT,ORIGPROT,PTID,SITE,VISCODE,EXAMDATE,DX_bl,AGE,PTGENDER,...,PTAU_bl,FDG_bl,PIB_bl,AV45_bl,FBB_bl,Years_bl,Month_bl,Month,M,update_stamp
0,2,ADNI1,ADNI1,011_S_0002,11,bl,2005-09-08,CN,74.3,Male,...,NaN,1.33615,NaN,NaN,NaN,0.000000,0.00000,0,0,2023-07-07 04:59:40
1,3,ADNI1,ADNI1,011_S_0003,11,bl,2005-09-12,AD,81.3,Male,...,22.83,1.10860,NaN,NaN,NaN,0.000000,0.00000,0,0,2023-07-07 04:59:40
2,3,ADNI1,ADNI1,011_S_0003,11,m06,2006-03-13,AD,81.3,Male,...,22.83,1.10860,NaN,NaN,NaN,0.498289,5.96721,6,6,2023-07-07 04:59:40
3,3,ADNI1,ADNI1,011_S_0003,11,m12,2006-09-12,AD,81.3,Male,...,22.83,1.10860,NaN,NaN,NaN,0.999316,11.96720,12,12,2023-07-07 04:59:40
4,3,ADNI1,ADNI1,011_S_0003,11,m24,2007-09-12,AD,81.3,Male,...,22.83,1.10860,NaN,NaN,NaN,1.998630,23.93440,24,24,2023-07-07 04:59:40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16416,4349,ADNI3,ADNI2,018_S_4349,18,m138,2023-03-30,CN,71.4,Female,...,17.82,1.33353,NaN,1.0344,NaN,11.351100,135.93400,138,138,2023-08-22 04:58:56
16417,6801,ADNI3,ADNI3,041_S_6801,41,m42,2023-06-30,SMC,61.0,Female,...,NaN,NaN,NaN,1.1509,NaN,3.701570,44.32790,42,42,2023-08-25 05:00:03
16418,5097,ADNI3,ADNI2,041_S_5097,41,m126,2023-08-16,SMC,67.5,Male,...,16.48,1.20863,NaN,1.1086,NaN,10.371000,124.19700,126,126,2023-08-26 05:00:28
16419,6515,ADNI3,ADNI3,007_S_6515,7,m60,2023-08-24,SMC,89.9,Female,...,NaN,NaN,NaN,NaN,1.1475,5.004790,59.93440,60,60,2023-08-29 04:58:51


# 1. Data Visualization & Understanding 

## 1.1 General Characteristics

In [6]:
# 1. SHAPE
print("1. SHAPE")
print(f"- Rows (visitas): {original_data.shape[0]}")
print(f"- Columns: {original_data.shape[1]}")
print(f"- Unique subjects (RID): {original_data['RID'].nunique()}")

# 2. VISITS BY SUBJECT
print("2. VISITS BY SUBJECT")
visits_per_subj = original_data.groupby("RID").size()
print(visits_per_subj.describe())

# 3. VISCODE - TYPES OF VISIT
print("3. VISCODE — TYPES OF VISIT")
print(original_data["VISCODE"].value_counts().head(20))

# 4. DIAGNOSTIC BASELINE (DX_bl) - (high potencial for classification !!!)
print("4. DIAGNOSTIC BASELINE (DX_bl) - (high potencial for classification)")
print(original_data["DX_bl"].value_counts(dropna=False))

# 5. DIAGNOSTIC BY VISITA (DX) - (for further conversion !!!)
print("5. DIAGNOSTIC BY VISITA (DX) - (for further conversion)")
print(original_data["DX"].value_counts(dropna=False))

# 6. MISSING VALUES
# 6.1 GENERAL MISSINGNESS
print("6.1 MISSINGNESS GERAL")
missing = original_data.isna().mean().sort_values(ascending=False) * 100
print(f"Columns with >50% missing values: {(missing > 50).sum()} de {len(missing)}")
print(f"Columns with 0% missing values: {(missing == 0).sum()}")

# 6.2 MISSINGNESS KEY-VARIABLES
print("6.2 MISSINGNESS - KEY VARIABLES (cognitive, imaging, biomarkers)")
key_vars = [
    "AGE", "PTGENDER", "PTEDUCAT", "APOE4",
    "CDRSB", "ADAS13", "MMSE", "MOCA", "FAQ",
    "Hippocampus", "WholeBrain", "Ventricles", "Entorhinal",
    "FDG", "AV45", "ABETA", "TAU", "PTAU",
]
key_vars = [v for v in key_vars if v in original_data.columns]
print(missing[key_vars].sort_values(ascending=False))

# 7. MISSINGNESS KEY-VARIABLES
print("7. DEMOGRAPHICS")
bl = original_data[original_data["VISCODE"] == "bl"]
print(f"Sujeitos com visita bl: {len(bl)}")
print("\nGénero:")
print(bl["PTGENDER"].value_counts())
print("\nAPOE4 (nº alelos):")
print(bl["APOE4"].value_counts(dropna=False))
print("\nIdade (baseline):")
print(bl["AGE"].describe())

1. SHAPE
- Rows (visitas): 16421
- Columns: 116
- Unique subjects (RID): 2430
2. VISITS BY SUBJECT
count    2430.000000
mean        6.757613
std         4.740714
min         1.000000
25%         3.000000
50%         5.000000
75%         9.000000
max        25.000000
dtype: float64
3. VISCODE — TYPES OF VISIT
VISCODE
bl      2430
m12     1970
m06     1618
m24     1596
m18     1320
m36     1070
m48      846
m30      815
m03      793
m60      466
m72      389
m42      357
m78      341
m66      328
m84      327
m96      267
m54      266
m90      250
m108     218
m120     148
Name: count, dtype: int64
4. DIAGNOSTIC BASELINE (DX_bl) - (high potencial for classification)
DX_bl
LMCI    5275
CN      4904
EMCI    2995
AD      1751
SMC     1485
NaN       11
Name: count, dtype: int64
5. DIAGNOSTIC BY VISITA (DX) - (for further conversion)
DX
MCI         4989
NaN         4963
CN          4020
Dementia    2449
Name: count, dtype: int64
6.1 MISSINGNESS GERAL
Columns with >50% missing values: 28 de 11

#### **ADNIMERGE - Initial Data Profile & Analysis**
1. **Dataset Structure**: 16,421 visit-records across 2,430 unique subjects (RID), 116 columns. Visit density is uneven and skewed: median of 5 visits/subject (IQR 3–9), but ranging from 1 to 25. All 2,430 subjects have a baseline (bl) visit; follow-up counts drop off steadily and non-monotonically at odd intervals (m03, m06, m18, m30, m42...), consistent with ADNI's evolving protocol across phases (ADNI1/GO/2/3). This has a direct modeling consequence: any split must be done at the subject (RID) level, not the row level, or you leak information across train/test via repeated visits of the same person 

2. **Diagnostic labels**: two candidate targets exist:
- `DX_bl` (baseline diagnosis): LMCI (5,275), CN (4,904), EMCI (2,995), AD (1,751), SMC (1,485 - 11 missing). This is a static, cross-sectional label, with one per subject-visit, fairly balanced across the five classes and nearly complete.
- `DX` (diagnosis per visit, i.e., over time): MCI (4,989), CN (4,020), Dementia (2,449 with 4,963 missing (~30%)). 

#### **ADNIMERGE - Missingness**
Of 116 columns, 28 (24%) exceed 50% missing; 18 are fully complete. This isn't uniform, it clusters by data modality:
| Tier    | Variables | Missing |
| ------- | ------- | ------- |
| Near-complete | Age, gender, education, APOE4 | 0–2% |
| Moderate | Cognitive tests (ADAS13, MMSE, FAQ, CDRSB) | 28–31% |
| High | Structural MRI (WholeBrain, Ventricles, Hippocampus, Entorhinal), MOCA | 40–55% |
| Very high | PET (FDG, AV45), CSF biomarkers (ABETA, TAU, PTAU) | 77–86% |

#### **ADNIMERGE - Demographics**
- Roughly balanced by **sex** (1,157 females; 1,273 males);
- **Age** centered at 72.9 (standard deviation 7.4, with range 50.4–91.4). An elderly cohort as expected for AD research, with limited spread at the young end. APOE4 allele count (0/1/2 copies) is well captured (only 2.2% missing), which is useful since it's a strong genetic risk marker and cheap to include.

## 1.2 Visualizatios